# Phase 0 — environment shakedown & experiment sizing
## FL layer-level communication scheduling: architecture survey

**What this notebook is.** A self-contained pre-flight for the planned joint
campaigns (ResNet50 + transformer stack, CIFAR-10, federated, from scratch).
It needs **only Python + TensorFlow + one GPU** — no Docker, no network
privileges, none of our code. It does four things:

1. checks your GPU/software environment,
2. runs our **granularity-tax calculator** — the analytical criterion this
   collaboration tests — on the campaign architectures *and on any model of
   yours you'd like to try*,
3. benchmarks local training throughput on your GPU,
4. sizes the full campaign from your measured numbers and emits a small
   JSON report to send back to us.

**What it is not.** The actual campaigns run in our containerized framework
(traffic-shaped links between containerized clients, data-integrity gating,
wall-clock measurement). That arrives as Docker images once Phase 0 looks
good — see the accompanying needs sheet.

**Background in one paragraph.** We measured that a per-layer deadline
mechanism cuts FL communication time by ~21%, but that sending *indivisible
layers* over parallel unequal-bandwidth links wastes ~24% on our small test
model — cancelling the gain. The waste is a pure function of the model's
layer-size table and the link speeds, so it can be computed without training.
Our calculator (independently reviewed; reproduces our measured numbers to
2e-4) predicts near-zero waste for ResNet50/MobileNetV2 and conditional waste
for transformers. The campaigns test those predictions — including the ones
where we expect our own mechanism to fail.

In [ ]:
# --- 1. Environment check -------------------------------------------------
import sys, json, time, shutil, subprocess, platform

report = {"python": sys.version.split()[0], "platform": platform.platform()}

import tensorflow as tf
report["tensorflow"] = tf.__version__
gpus = tf.config.list_physical_devices("GPU")
report["gpus"] = [g.name for g in gpus]
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True,
                         text=True, timeout=10)
    report["nvidia_smi"] = smi.stdout.strip().splitlines()
except Exception as e:
    report["nvidia_smi"] = f"unavailable ({e})"

# Informational only — the campaigns (not this notebook) will need these:
report["docker_present"] = shutil.which("docker") is not None
report["tc_present"] = shutil.which("tc") is not None

print(json.dumps(report, indent=2))
assert gpus, "No GPU visible to TensorFlow — fix this before anything else."

## 2. The granularity-tax calculator

The model: each client uploads its per-layer updates over parallel links of
unequal bandwidth (our testbed: three links, speed ratio **6:3:1**). Layers
are indivisible, so the scheduler places whole layers on links
(greedy earliest-finish-time, largest first — a faithful replica of our
framework's scheduler, verified against it on 4,000 randomized inputs with
zero mismatches). The **tax** is the finish time of the best whole-layer
schedule divided by the perfectly-divisible ideal (total bytes / total
bandwidth). Tax ≈ 1.00 → granularity costs nothing. Our measured system
value on our own CNN is **1.312**; the didactic example below shows the
mechanism in its purest form.

Only the **ratio** of link speeds matters — the tax is exactly invariant to
scaling all bandwidths together (verified), so results here transfer to any
absolute link speed.

In [ ]:
# --- 2. Tax calculator (self-contained) -----------------------------------
import math

CLASSES = {0: 6.0, 1: 3.0, 2: 1.0}   # relative link speeds (our testbed's ratio)

def eft_makespan(sizes, classes=CLASSES):
    """Whole-layer schedule length: greedy earliest-finish-time, largest first."""
    loads = {c: 0.0 for c in classes}
    for s in sorted(sizes, reverse=True):
        c = min(sorted(classes), key=lambda c: loads[c] + s / classes[c])
        loads[c] += s / classes[c]
    return max(loads.values())

def tax(sizes, classes=CLASSES):
    fluid = sum(sizes) / sum(classes.values())   # perfectly-divisible ideal
    return eft_makespan(sizes, classes) / fluid

def tensor_sizes_bytes(model, overhead=90):
    """Per-tensor wire sizes of a keras model (4 B/param + framing overhead)."""
    return [int(math.prod(w.shape)) * 4 + overhead for w in model.trainable_weights]

def describe(name, sizes):
    sizes = sorted(sizes, reverse=True)
    total = sum(sizes)
    print(f"{name:24s} tensors={len(sizes):4d}  payload={total/1e6:8.2f} MB  "
          f"top5={sum(sizes[:5])/total*100:5.1f}%  TAX={tax(sizes):6.3f}")

# Didactic check — the failure mode in its purest form:
# five equal indivisible chunks on 6:3:1 links. Balanced finish needs
# 3 / 1.5 / 0.5 chunks — impossible with whole chunks; best schedule is 4/3
# of the ideal. (Our real CNN measures 1.312: same mechanism, plus crumbs.)
five_equal = [100] * 5
print(f"didactic 5-equal-chunks tax = {tax(five_equal):.4f}  (expect 1.3333)")
assert abs(tax(five_equal) - 4/3) < 1e-9

In [ ]:
# --- 2b. The criterion on the campaign architectures (and yours) ----------
# weights=None builds architecture only: no downloads, no training.
resnet = tf.keras.applications.ResNet50(weights=None,
                                        input_shape=(32, 32, 3), classes=10)
mobile = tf.keras.applications.MobileNetV2(weights=None,
                                           input_shape=(32, 32, 3), classes=10)
describe("ResNet50 (32x32)", tensor_sizes_bytes(resnet))
describe("MobileNetV2 (32x32)", tensor_sizes_bytes(mobile))

# Expected: both ~1.00 — many graded tensor sizes pack almost perfectly.
# Contrast (from our records): our 14-tensor CNN = 1.312 measured;
# VGG16 = 1.238 (one 411 MB tensor is 74% of the model).
#
# IMPORTANT CAVEAT we will test, not assume: these are full-model values.
# Under active layer-shedding the tax can RISE (transformers: from ~1.005
# to 1.2-1.5 in our swept computation, because their large tensors are
# identical and lose all balancing slack). That is exactly why the
# transformer campaign is pre-registered as our predicted-failure case.

# >>> Try your own model here — this is the criterion the survey validates:
# my_model = ...  # any tf.keras model
# describe("my model", tensor_sizes_bytes(my_model))

## 3. Local-training throughput benchmark

The campaigns' GPU cost is dominated by clients' local training. We measure
one number per architecture on your hardware — **seconds per full-dataset
epoch** — and size everything from it. This cell trains briefly (2 epochs;
the first is warm-up/compilation and is discarded). Accuracy here is
irrelevant; only throughput matters.

*Note:* the benchmark uses the stock ResNet50 for timing. The campaign
model ships with our framework and differs in two standard FL adaptations
(GroupNorm instead of BatchNorm — federated averaging corrupts BatchNorm's
running statistics — and a CIFAR stem); their throughput impact is minor.

In [ ]:
# --- 3. Throughput benchmark ----------------------------------------------
(x, y), _ = tf.keras.datasets.cifar10.load_data()
x = x.astype("float32") / 255.0

def build_transformer(d=256, blocks=6, heads=8, patch=4):
    """Compact ViT-style classifier — timing proxy for the transformer cell."""
    inp = tf.keras.Input((32, 32, 3))
    t = tf.keras.layers.Conv2D(d, patch, strides=patch)(inp)   # patchify
    t = tf.keras.layers.Reshape((-1, d))(t)
    for _ in range(blocks):
        a = tf.keras.layers.LayerNormalization()(t)
        a = tf.keras.layers.MultiHeadAttention(heads, d // heads)(a, a)
        t = t + a
        m = tf.keras.layers.LayerNormalization()(t)
        m = tf.keras.layers.Dense(4 * d, activation="gelu")(m)
        m = tf.keras.layers.Dense(d)(m)
        t = t + m
    t = tf.keras.layers.GlobalAveragePooling1D()(t)
    return tf.keras.Model(inp, tf.keras.layers.Dense(10)(t))

def sec_per_epoch(model, batch=128):
    model.compile("adam",
                  tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
    model.fit(x, y, batch_size=batch, epochs=1, verbose=0)      # warm-up
    t0 = time.time()
    model.fit(x, y, batch_size=batch, epochs=1, verbose=0)      # measured
    return time.time() - t0

vit = build_transformer()
describe("Transformer (bench)", tensor_sizes_bytes(vit))

timings = {}
for name, m in [("resnet50", resnet), ("transformer", vit)]:
    timings[name] = round(sec_per_epoch(m), 1)
    print(f"{name:12s} {timings[name]:7.1f} s / full-dataset epoch")
report["sec_per_epoch"] = timings

In [ ]:
# --- 4. Campaign sizing from YOUR measured throughput ----------------------
# Pre-registered design, per architecture:
#   3 arms (baseline / shedding / blind-control) x 6 seeds = 18 runs
#   100 rounds; per round each of 3 clients trains 1 local epoch on 1/3 of
#   the data  =>  ~1 full-dataset epoch-equivalent of GPU work per round.
ARMS_X_SEEDS, ROUNDS = 18, 100
FINE_TUNE_FRACTION = 0.3          # short fine-tuning variant, ~30% extra
N_GPUS = max(1, len(gpus))

total_h = 0.0
print(f"{'campaign':14s} {'GPU-hours':>10s}   (runs x rounds x sec/epoch)")
for name, spe in timings.items():
    h = ARMS_X_SEEDS * ROUNDS * spe * (1 + FINE_TUNE_FRACTION) / 3600
    total_h += h
    print(f"{name:14s} {h:10.1f}")
print(f"{'TOTAL':14s} {total_h:10.1f}  GPU-hours "
      f"(~{total_h/24/N_GPUS:.1f} days on your {N_GPUS} visible GPU(s))")
report["campaign_gpu_hours"] = round(total_h, 1)

# Storage: partitions + logs + result mirrors, generously rounded.
print("scratch storage needed: ~200 GB for the campaign duration")

In [ ]:
# --- 5. Environment report — please send us this file ----------------------
with open("phase0_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))
print("\nWritten to phase0_report.json — send it back to us and we ship the\n"
      "containerized campaign (Docker images + configs + pre-registered\n"
      "prediction document) sized to these numbers.")

## What happens after Phase 0

1. **You send us `phase0_report.json`.** It contains versions, GPU inventory,
   measured throughput, and the derived campaign size — nothing else.
2. **We ship the campaign as Docker images** plus configs and the
   pre-registered prediction document (signed off by you before anything
   runs). Host requirements at that point: Docker + NVIDIA container
   toolkit, ~200 GB scratch, and *either* permission for traffic shaping
   (`tc`, NET_ADMIN) inside containers *or* the fallback split where your
   GPUs handle training and network emulation stays on our side.
3. **We operate the campaigns and pull results continuously**; your
   involvement can be limited to access, or extend to the science —
   scenario design and the real-network validation experiment (see the
   needs sheet, items B and C).

**The stake, stated up front:** the survey's boldest pre-registered
prediction is that even on ResNet50 — the first geometry where
importance-based layer selection has room to beat blind selection — it
still won't. If that holds, geometry-dominates-policy becomes a strong
general claim; if it fails, that's a discovery too. Your hardware decides.